# Q-Phase · Question

Problem, Forschungsfrage, Zielgruppen, Nutzenversprechen und Erfolgskriterien.

**Projekt:** MietCheck · Big Data & Data Analytics · QUA³CK

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COLORS = {"navy": "#14213D", "blue": "#2563EB", "teal": "#0F766E",
          "amber": "#F59E0B", "red": "#DC2626", "grey": "#64748B"}
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.titleweight": "bold",
                     "axes.labelsize": 10, "figure.dpi": 110})

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

print(f"Projektwurzel: {ROOT}")

Projektwurzel: C:\Users\nelek\Desktop\Big Data\MietCheck


## 1. Zentrale Forschungsfrage

> **Wie groß ist die Lücke zwischen kleinräumiger Bestandsmiete und aktueller Angebotsmiete in Deutschland, welche räumlichen und wohnungsbezogenen Faktoren erklären die Bestandsmiete und was bedeutet der Abstand für eine konkrete Wohnsituation?**

MietCheck trennt drei Größen, die herkömmliche Rechner häufig vermischen:

1. den modellierten lokalen **Bestandsanker** aus dem Zensus 2022,
2. den aktuellen **Angebotsmarkt** aus GREIX 2026-Q1,
3. die **persönliche Vertragsmiete und Mietbelastung**.

Die Differenz aus Angebot und Bestand heißt in der App **Umzugsaufschlag**. Sie ist ein deskriptiver Vergleich, keine zulässige oder „faire“ Miete.

## 2. Thematische Forschungsfragen

1. Wie stark variieren Zensus-Bestandsmieten innerhalb und zwischen Regionen?
2. Verbessern räumliche und strukturelle Merkmale eine fachliche Kategorien-Baseline?
3. Welches klassische Regressionsverfahren generalisiert auf unbekannte räumliche Gebiete?
4. Wie wirken Skalierung und Algorithmuswahl bei Ridge, SVM, Bäumen und neuronalen Netzen?
5. Wie groß ist der Abstand zum aktuellen GREIX-Angebotsmarkt?
6. Wie lässt sich Modellunsicherheit sichtbar und ohne Scheingenauigkeit kommunizieren?

## 3. Zielgruppe und Anwendungsnutzen

| Zielgruppe | Entscheidung | Mehrwert in der App |
|---|---|---|
| Bestandsmietende | Bleiben oder umziehen? | eigene Miete vs. Bestand vs. aktuelles Angebot |
| Wohnungssuchende | Welche Mehrkosten sind plausibel? | Umzugsaufschlag in €/m², Euro und Prozent |
| Studierende/Berufseinsteigende | Wie verändert sich die Belastung? | Anteil am Haushaltsnetto |
| Dateninteressierte | Wie unterscheiden sich Märkte? | Zeitreihe, IQR, Methodik und Grenzen |

## 4. Datenbasis, Scope und Grenzen

- **ML-Ziel:** durchschnittliche Nettokaltmiete am 15.05.2022 im 100-m-Zensusgitter
- **Aktueller Markt:** GREIX-Angebotsmieten bis 2026-Q1, methodisch getrennt vom Training
- **Geografischer Scope:** Deutschland; aktuelle Marktwerte nur für belegte GREIX-Regionen
- **Keine Behauptung:** keine adressgenaue Wohnungsschätzung, kein Mietspiegel, keine Rechts- oder Finanzberatung
- **Datenschutz:** persönliche Eingaben werden nicht gespeichert

In [2]:
build = load_json("reports/dataset_build_report.json")
greix = load_json("data/app/greix_metadata.json")

scope = pd.DataFrame([
    ("Zensus-Modellzeilen", f"{build['rows']:,}"),
    ("eindeutige 100-m-Zellen", f"{build['unique_grid_cells']:,}"),
    ("25-km-Raumblöcke", build["spatial_blocks_25km"]),
    ("GREIX-Regionen", greix["regions"]),
    ("GREIX-Datenstand", greix["latest_period"]),
], columns=["Scope-Nachweis", "Wert"])
display(scope.style.hide(axis="index"))

Scope-Nachweis,Wert
Zensus-Modellzeilen,"2,058,569"
eindeutige 100-m-Zellen,"1,184,386"
25-km-Raumblöcke,663
GREIX-Regionen,38
GREIX-Datenstand,2026-Q1


## 5. Marktumfeld und Alleinstellungsmerkmal

Destatis beschreibt den Bestand, Immobilienportale aktuelle Angebote, kommunale Mietspiegel einen rechtlich definierten lokalen Vergleich und Budgetrechner die persönliche Belastung. MietCheck verbindet erstmals in einer **quelloffenen, reproduzierbaren Nutzerreise**:

**kleinräumiges ML-Bestandsmodell + aktueller unabhängiger Angebotsmarkt + persönliche Mietbelastung + zwei getrennte Unsicherheitsarten.**

Der Markt-IQR beschreibt die Streuung heutiger Angebote; das conformale Modellband beschreibt den Fehler des Zensusmodells. Beide werden niemals vermischt.

## 6. Erfolgskriterien vor dem finalen Test

| Dimension | Go/No-Go-Regel |
|---|---|
| Big Data | mindestens 2 Mio. reproduzierbare Modellzeilen |
| A¹ Auswahl | klassische Baseline plus lineare, SVM-, Baum-, Ensemble- und NN-Verfahren |
| Generalisierung | räumlich disjunkte Entwicklung, Kalibrierung und Test |
| Nutzen | mindestens 15 % niedrigerer Test-MAE als Kategorienmedian |
| Unsicherheit | separates Kalibrierungsset und empirisch gemessene Coverage |
| Reproduzierbarkeit | Download, Hashes, Tests, CI und ausgeführte Notebooks |
| Produkt | transparente Streamlit-App mit Quellen, Datenstand und Grenzen |

In [3]:
benchmark = load_json("reports/algorithm_benchmark.json")
final = load_json("reports/final_model_evaluation.json")
checks = pd.DataFrame([
    ("Big Data", build["rows"] >= 2_000_000, build["rows"]),
    ("mind. 7 Kandidaten inkl. Baseline", len(benchmark["models"]) >= 7, len(benchmark["models"])),
    ("MAE-Verbesserung ≥ 15 %", final["test"]["mae_improvement_vs_baseline"] >= .15,
     final["test"]["mae_improvement_vs_baseline"]),
    ("separate Kalibrierung", final["partition"]["calibration"]["blocks"] > 0,
     final["partition"]["calibration"]["blocks"]),
], columns=["Kriterium", "erfüllt", "Nachweis"])
display(checks.style.hide(axis="index"))

Kriterium,erfüllt,Nachweis
Big Data,True,2058569.000000
mind. 7 Kandidaten inkl. Baseline,True,7.000000
MAE-Verbesserung ≥ 15 %,True,0.383298
separate Kalibrierung,True,99.000000


## 7. Web-App-Konzept

Die K-Phase übersetzt das Modell in drei App-Tabs: **Dein Mietbild**, **Marktverlauf** und **Methodik & Quellen**. Nutzerinnen und Nutzer wählen Region, Fläche und Baualtersklasse und können eigene Miete sowie Einkommen ergänzen. Ausgegeben werden Bestandsanker, Modellband, GREIX-Median, Markt-IQR, Umzugsaufschlag und persönliche Belastung.

---

**Reproduzierbarkeit:** Kennzahlen stammen aus versionierten JSON-/CSV-Artefakten. Die genannten Skripte erzeugen sie aus den öffentlichen Rohdaten erneut. Schwere Trainingsläufe liegen bewusst in getesteten Skripten, damit Notebook und produktive Pipeline dieselben Splits, Parameter und Reports verwenden.